## 06 - Topic model training

Trains a BERTopic model on the Italian reviews and saves it with the topic of each review.
A new model can number and size its topics differently, so after a run the label map in 07 may need to be rebuilt.

Input: `data/processed/trenord_reviews_clean.csv`

Output: `data/processed/trenord_reviews_topics.csv` and `data/processed/bertopic_model_trained/`

In [1]:
# --- IMPORTS ---
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.it.stop_words import STOP_WORDS as italian_stopwords

# --- LOAD DATA ---
df = pd.read_csv("../data/processed/trenord_reviews_clean.csv")
df["date"] = pd.to_datetime(df["date"])

print(f"Loaded {len(df)} reviews")
print(f"Language distribution:\n{df['language'].value_counts()}")

W1105 12:45:04.008000 38856 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loaded 7329 reviews
Language distribution:
language
it         6951
unknown     363
en            5
la            5
fr            2
es            1
mi            1
pt            1
Name: count, dtype: int64


In [2]:
# --- PREPARE TEXTS ---
# Filter for Italian reviews because we want to analyze them separately.
italian_df = df[df["language"] == "it"].copy()

# We use text_clean (not text_lemmatized) because BERTopic uses transformer
# embeddings that work better with natural text. BERTopic handles
# tokenization and stopwords internally.
texts = italian_df["text_clean"].tolist() # tolist() converts the pandas Series to a list of strings

# BERTopic cannot handle empty strings, so we replace them with a single space " ".
# We check two conditions for each text:
# - it must be a string (not NaN)
# - it must contain at least one non-space character
for i in range(len(texts)):
    text = texts[i] # Get the text at index i
    is_valid = isinstance(text, str) and len(text.strip()) > 0
    if not is_valid:
        texts[i] = " "

# Count how many texts ended up empty after the replacement above
empty_count = 0
for text in texts:
    if text.strip() == "":
        empty_count += 1

print(f"Italian reviews : {len(italian_df)}")
print(f"Empty texts     : {empty_count}")
print(f"Texts prepared  : {len(texts)}")

Italian reviews : 6951
Empty texts     : 0
Texts prepared  : 6951


In [3]:
# --- CONFIGURE BERTOPIC ---
# BERTopic works in 4 steps:
# 1. EMBEDDING : converts texts into numerical vectors
# 2. UMAP      : reduces vector dimensions for clustering
# 3. HDBSCAN   : groups similar texts into topics
# 4. c-TF-IDF  : finds the most representative words per topic
# On top of these, we add a representation model (MMR) to refine topic keywords.

# Step 1 - Embedding model
# paraphrase-multilingual-mpnet-base-v2 is the best lightweight
# multilingual model that runs on CPU and handles Italian well
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# Step 2 - UMAP for dimensionality reduction
# random_state=42 is critical - without it results change every run
umap_model = UMAP(
    n_neighbors=15,   # how many neighbors each point considers
    n_components=5,   # reduce to 5 dimensions before clustering
    min_dist=0.0,     # pack cluster points tightly together
    metric="cosine",  # best distance metric for text embeddings
    random_state=42   # makes results reproducible
)

# Step 3 - HDBSCAN for clustering
# min_cluster_size=30 means a topic needs at least 30 reviews to be valid
hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    metric="euclidean",
    prediction_data=True  # needed to assign topics to new documents
)

# Step 4 - CountVectorizer with Italian stopwords and bigrams
# We define this here so we use it once in update_topics later
# ngram_range=(1,2) means we consider both single words and two-word phrases
# BERTopic fits the vectorizer on one joined text per topic, so min_df counts topics, not reviews.
# It changes only the keywords, not which reviews belong to which topic (compared in 05).
vectorizer_model = CountVectorizer(
    stop_words=list(italian_stopwords),  # remove Italian stopwords
    min_df=5,                            # keyword must appear in at least 5 topics
    ngram_range=(1, 2)                   # single words + two-word phrases
)

# Step 5 - Representation model: Maximal Marginal Relevance (MMR)
# By default, c-TF-IDF may return redundant keywords that mean the same thing,
# such as "biglietto", "biglietti", "acquisto biglietti".
# MMR re-ranks the candidate keywords to maximize diversity, so the final
# topic representation contains less redundant and more informative words.
# diversity ranges from 0 (no diversity) to 1 (maximum diversity).
# We use 0.3 - the value recommended in the official BERTopic documentation -
# which reduces redundancy while keeping the keywords relevant to the topic.
representation_model = MaximalMarginalRelevance(diversity=0.3)

# Combine everything into BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,        # Italian stopwords and bigrams
    representation_model=representation_model, # MMR for diverse topic keywords
    top_n_words=10,
    language="multilingual",
    calculate_probabilities=False,  # faster, we don't need soft assignments
    verbose=True
)

print("BERTopic configured and ready.")

Loading weights: 100%|██████████| 199/199 [00:02<00:00, 75.40it/s]


BERTopic configured and ready.


In [4]:
# --- TRAIN BERTOPIC ---
# fit_transform() trains the model AND assigns a topic to each text.
# topic -1 means "outlier" - a review that doesn't fit any topic clearly.

print("Training BERTopic... this may take several minutes on CPU.")

# We ignore the second returned value (probabilities) because we set
# calculate_probabilities=False, so it would be None anyway.
topics, _ = topic_model.fit_transform(texts)

# Store the topic assignments in the dataframe
italian_df["topic"] = topics

# Count how many reviews fall into each topic.
# value_counts() returns a Series: index = topic id, value = number of reviews.
topic_counts = italian_df["topic"].value_counts()

# Count outliers - reviews assigned to the special topic -1
n_outliers = topic_counts.get(-1, 0)  # .get avoids an error if there are no outliers

# Count real topics - all unique topic ids except the outlier topic -1
unique_topics = set(topics)
unique_topics.discard(-1)             # remove -1 if present; does nothing if absent
n_topics = len(unique_topics)

# Count assigned reviews - total reviews minus the outliers
n_assigned = len(topics) - n_outliers

print(f"\nTraining complete!")
print(f"Topics found : {n_topics}")
print(f"Outliers     : {n_outliers}")
print(f"Assigned     : {n_assigned}")

2026-11-05 12:45:19,687 - BERTopic - Embedding - Transforming documents to embeddings.


Training BERTopic... this may take several minutes on CPU.


Batches: 100%|██████████| 218/218 [02:42<00:00,  1.34it/s]
2026-11-05 12:48:01,888 - BERTopic - Embedding - Completed ✓
2026-11-05 12:48:01,889 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-05 12:48:29,263 - BERTopic - Dimensionality - Completed ✓
2026-11-05 12:48:29,264 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-05 12:48:29,425 - BERTopic - Cluster - Completed ✓
2026-11-05 12:48:29,430 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-05 12:48:35,806 - BERTopic - Representation - Completed ✓



Training complete!
Topics found : 30
Outliers     : 1601
Assigned     : 5350


In [5]:
# --- REDUCE OUTLIERS ---
# We assign each outlier to its closest topic using embedding similarity.
# strategy="embeddings" : uses cosine similarity between review and topic centroid
# threshold=0.0 : assigns ALL outliers - we inspected them and they are valid reviews

# Count outliers before reduction, straight from the current topic column
outliers_before = (italian_df["topic"] == -1).sum()

# Reassign outliers to their closest topic
new_topics = topic_model.reduce_outliers(
    texts,
    topics,
    strategy="embeddings",
    threshold=0.0
)
italian_df["topic"] = new_topics

# Count outliers after reduction - should be 0 since threshold=0.0 assigns all of them
outliers_after   = (italian_df["topic"] == -1).sum()
reviews_assigned = (italian_df["topic"] != -1).sum()

print(f"Outliers before : {outliers_before}")
print(f"Outliers after  : {outliers_after}")
print(f"Reviews assigned: {reviews_assigned}")

Outliers before : 1601
Outliers after  : 0
Reviews assigned: 6951


In [6]:
# --- UPDATE TOPIC REPRESENTATIONS ---
# After reducing outliers, we recompute the topic representations so they
# reflect the new assignments. We pass two models explicitly:
# 1. vectorizer_model     : applies Italian stopwords and bigrams
# 2. representation_model : applies MMR to keep topic keywords diverse
# Passing both here ensures the final representations are consistent with
# the configuration we set when building the model.

topic_model.update_topics(
    texts,
    topics=new_topics,                         # the updated topic assignments after outlier reduction
    vectorizer_model=vectorizer_model,         # Italian stopwords, bigrams and min_df=5
    representation_model=representation_model  # MMR - keeps keywords diverse
)

# Retrieve the updated topic information
topic_info = topic_model.get_topic_info()

# Keep only real topics - exclude the outlier topic -1
real_topics = topic_info[topic_info["Topic"] != -1]

print(f"Total topics : {len(real_topics)}")
print()
for _, row in real_topics.iterrows():
    print(f"Topic {row['Topic']:3d} | {row['Count']:4d} reviews | {row['Name']}")

2026-11-05 12:49:23,123 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Total topics : 30

Topic   0 | 1214 reviews | 0_treni_treno_ritardi_app
Topic   1 | 1143 reviews | 1_biglietti_acquistare biglietti_app_acquisto
Topic   2 |  641 reviews | 2_pagamento_abbonamenti_rinnovare abbonamento_credito
Topic   3 |  499 reviews | 3_lenta_inutile_inaffidabile_difficile
Topic   4 |  515 reviews | 4_app_aggiornamento_telefono_android
Topic   5 |  219 reviews | 5_login_account_accesso_errore
Topic   6 |  185 reviews | 6_ritardi_ritardo_ritardi cancellazioni_arrivano ritardo
Topic   7 |  173 reviews | 7_stazioni_stazione_stazioni preferite_aggiornamento
Topic   8 |  176 reviews | 8_ottima app_app comoda_app facile_app funziona
Topic   9 |  152 reviews | 9_ottima applicazione_applicazione_utilissima_applicazione utile
Topic  10 |  164 reviews | 10_ottima_perfetta_perfetto_fantastica
Topic  11 |  145 reviews | 11_facile_semplice_facile usare_intuitivo
Topic  12 |  175 reviews | 12_biglietti_acquisto biglietti_acquistare biglietti_app
Topic  13 |  118 reviews | 13_biglie

In [7]:
# --- HOW BERTOPIC BUILDS THE AUTOMATIC TOPIC NAMES ---

# 1. The topic table returned by get_topic_info(): one row per topic
print("Columns:", list(topic_info.columns))

# The first 5 topics. Representative_Docs is left out on purpose:
# it holds real review texts, and notebook outputs are published.
display(topic_info[["Topic", "Count", "Name", "Representation"]].head(5))

# 2. Where the Name column comes from: the topic number followed by
# its first 4 keywords, joined with underscores. Topic 0 as an example.

# The 10 keywords of topic 0 (get_topic returns word and weight pairs, we keep the word)
keywords = []
for word, weight in topic_model.get_topic(0):
    keywords.append(word)
print("All 10 keywords :", keywords)

# The first 4 keywords are the ones used in the name
print("First 4 keywords:", keywords[:4])

# The automatic name, as stored by BERTopic
print("Automatic name  :", topic_model.topic_labels_[0])

# The same name rebuilt by hand: "0" plus the first 4 keywords, joined with "_"
rebuilt_name = "0_" + "_".join(keywords[:4])
print("Rebuilt by hand :", rebuilt_name)

Columns: ['Topic', 'Count', 'Name', 'Representation', 'Representative_Docs']


,Topic,Count,Name,Representation
0,0,1214,0_treni_treno_ritardi_app,"[treni, treno, ritardi, app, stazione, servizi..."
1,1,1143,1_biglietti_acquistare biglietti_app_acquisto,"[biglietti, acquistare biglietti, app, acquist..."
2,2,641,2_pagamento_abbonamenti_rinnovare abbonamento_...,"[pagamento, abbonamenti, rinnovare abbonamento..."
3,3,499,3_lenta_inutile_inaffidabile_difficile,"[lenta, inutile, inaffidabile, difficile, inco..."
4,4,515,4_app_aggiornamento_telefono_android,"[app, aggiornamento, telefono, android, proble..."


All 10 keywords : ['treni', 'treno', 'ritardi', 'app', 'stazione', 'servizio', 'orari treni', 'treni ritardo', 'cancellazioni', 'aggiornata']
First 4 keywords: ['treni', 'treno', 'ritardi', 'app']
Automatic name  : 0_treni_treno_ritardi_app
Rebuilt by hand : 0_treni_treno_ritardi_app


In [8]:
# --- KEYWORDS OF THE TOPICS AS BAR CHARTS ---
# The top 5 keywords of the 8 largest topics. A longer bar means a higher
# c-TF-IDF weight, so a word more typical of that topic.
fig = topic_model.visualize_barchart(top_n_topics=8, n_words=5, width=400, height=300)
fig.show()

The top 5 keywords of the 8 largest topics. A longer bar means a higher c-TF-IDF weight, so a word more typical of that topic. The keywords match the labels given in 07: topic 0 is about delays, 1 about tickets, 2 about subscriptions, 5 about login. Topic 7 (173 reviews) is not shown because topic 8 is slightly larger (176).

In [9]:
# --- HOW SIMILAR THE TOPICS ARE ---
# A tree of the topics: topics joined further to the left are more similar to each other.
# It supports the merges in 07, for example topics 0, 6 and 29 should end up close.
fig = topic_model.visualize_hierarchy()
fig.show()

Topics joined further to the left are more similar. The tree supports several merges made in 07: 0, 6 and 29 (delays), 5 and 22 (login and registration), and 8, 11, 20 and 24 (ease of use).

Similarity here is computed on the keywords, not on what users think. Topic 12 (praise for buying tickets) sits next to topic 1 (failed purchases), because both use words like "biglietti" and "acquisto". For this reason the merges in 07 also rely on ratings and on reading the reviews, not on this tree alone.

Labels are cut at 30 characters by BERTopic. The full names are in the topic list printed above.

In [10]:
# --- CHECK BEFORE SAVING ---
# A last look at what is about to be saved, so a problem in an earlier cell
# is caught before the files are overwritten.
# assert stops the notebook with the message shown if the condition is false.

# Every Italian review has a topic
assert "topic" in italian_df.columns, "The topic column is missing"
assert len(italian_df) == len(texts), "The table and the texts have a different number of rows"

# No review is left as an outlier after the reduction
n_outliers = (italian_df["topic"] == -1).sum()
assert n_outliers == 0, f"{n_outliers} reviews still have topic -1"

# The table and the model have the same number of topics
n_topics_table = italian_df["topic"].nunique()
n_topics_model = len(real_topics)
assert n_topics_table == n_topics_model, "The table and the model have a different number of topics"

print(f"Reviews to save: {len(italian_df):,}")
print(f"Topics         : {n_topics_table}")
print(f"Outliers       : {n_outliers}")
print("All checks passed")

Reviews to save: 6,951
Topics         : 30
Outliers       : 0
All checks passed


In [11]:
# --- SAVE MODEL AND DATA ---
# The model is saved in safetensors format instead of pickle. A pickle file can run
# code when it is loaded; safetensors stores only numbers, so it is safer and smaller.
# With safetensors, BERTopic keeps topics, keywords and assignments, but not the UMAP
# and HDBSCAN models: a reloaded model assigns new reviews by similarity to the topics.
# That is enough here, because 07 only works on the reviews already assigned.
embedding_model_name = "paraphrase-multilingual-mpnet-base-v2"

topic_model.save(
    "../data/processed/bertopic_model_trained",
    serialization="safetensors",
    save_ctfidf=True,                        # word counts, so keywords can be recomputed (07 merges topics)
    save_embedding_model=embedding_model_name  # only the model name: it is reloaded by name
)
print("BERTopic model saved in safetensors format.")

# The reviews with their topic go to a new file, not back into the cleaning output,
# so every step of the pipeline keeps its own output:
# - 03_cleaning.ipynb             → trenord_reviews_clean.csv  (cleaned text, no topics)
# - 06_topic_model_training.ipynb → trenord_reviews_topics.csv (adds the topic of each review)
# Labels are added later, in 07.
italian_df.to_csv(
    "../data/processed/trenord_reviews_topics.csv",
    index=False,
    encoding="utf-8-sig"
)
print(f"Dataframe saved: {len(italian_df):,} reviews with topic assignments.")

BERTopic model saved in safetensors format.
Dataframe saved: 6,951 reviews with topic assignments.
